In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

df = sns.load_dataset('titanic')

print("=== 결측치 처리 및 병합===")
print("1. 결측치 확인")
print(df.isnull().sum()[df.isnull().sum()>0])

df_clean = df.copy()
#중앙값 대체
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())
#최빈값 대체
df_clean['embarked'] = df_clean['embarked'].fillna(df_clean['embarked'].mode()[0])
# 결측치 과다로 삭제
df_clean = df_clean.drop(columns=['deck'])

print("==정제 후 결측치 확인==")
print(df_clean[['age','embarked']].isnull().sum())

df_a = pd.DataFrame({'id' : [1, 2, 3] , 'name' : ['kim','lee','park']})
df_b = pd.DataFrame({'id' : [1,2, 3], 'score' : [90, 85, 95]})

merged_inner = pd.merge(df_a, df_b, on = 'id', how = 'inner')

print ("=== inner join 확인 ===")
print(merged_inner)

In [ ]:
print("== 파생변수 ==")

df_clean['family_size'] = df_clean['sibsp'] + df_clean['parch'] + 1

df_clean['is_alone'] = (df_clean['family_size'] == 1).astype(int)
# 연속 수치형 데이트를 가지고 범주형 데이터로 구간화
bins = [0, 12, 19, 35, 60 ,100]
labels = ['아동', '청소년', '청년', '중년', '노년']
df_clean['age_group'] = pd.cut(df_clean['age'] , bins= bins, labels=labels)

print(df_clean[['age', 'age_group', 'family_size', 'is_alone']].head())

In [ ]:
#그룹별 집계

#1. 단일 변수 그룹화
pclass_summary = df_clean.groupby('pclass')[['fare','survived']].mean()
print("객실 등급별 편균 운임 및 생존율")
print(pclass_summary)
print()
# 2. 다중 변수 그룹화
grouped_result = df_clean.groupby(['sex','pclass']).agg(
    승객수 =('survived','count'),
    생존율 =('survived','mean'),
    평균나이=('age','mean'),
    최대운임=('fare','max')
).reset_index()

print("성별, 객실 등급별 요약")
grouped_result

데이터프레임 병합(merge & concat)

In [ ]:
#병합

passenger_info = pd.DataFrame({
    'passenger_id':[1,2,3,4],
    'name':['kim','lee','park','choi'],
    'pclass':[1,2,1,3],
})
passenger_fare = pd.DataFrame({
    'passenger_id':[1,2,3,5],
    'fare':[100.0,50.0,80.0,30.0]
})
# inner join : 교집합, on = 연결키, how= 조인 방법
inner_merged = pd.merge(passenger_info, passenger_fare, on = 'passenger_id', how = 'inner')
print("inner join") #공통적인 key를 가져서 합치는거  여기서는 passenger_id를 기준으로 <<
print(inner_merged)
print()
#left join : 왼쪽 테이블 기준 (왼쪽 테이블 = passenger_info를 기준)으로
left_merged = pd.merge(passenger_info, passenger_fare, on = 'passenger_id', how = 'left')
print("left join")
print(left_merged)

#pd.concat(): 행/열 방향 단순 결합
df_part1 = df_clean.iloc[0:3, :3]
df_part2 = df_clean.iloc[3:6, :3]

concat_rows = pd.concat([df_part1, df_part2], axis = 0)
print("행 결합")
print(df_part1)
print(df_part2)
print(concat_rows)

concat_cols = pd.concat([df_part1, df_part2], axis = 1)

머신러닝 데이터 전처리 파이프라인 구축 (자동화)
* 전처리 순서
1. 데이터 탐색(EDA)및 선택
2. 결측치 정제
3. 이상치 정제
4. 인코딩
5. 스케일링

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler

def pipeline_titanic_preprocessing(raw_df):
  df = raw_df[['pclass', 'sex', 'age', 'fare', 'embarked']].copy()

  #step: 결측치 처리
  df['age'] = df['age'].fillna(df['age'].median())
  df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
  #step : 이상치 처리
  # 요금에 이상치를 IQR의 상한선으로 대
  Q1 = df['fare'].quantile(0.25)
  Q3 = df['fare'].quantile(0.75)
  IQR = Q3 - Q1
  upper_bound = Q3 + (1.5 * IQR)
  lower_bound = Q1 - (1.5 * IQR)
  df.loc[df['fare']> upper_bound, 'fare'] = upper_bound
  # step: 수치 스케일링
  # age와 fare 수치를 StandardScaler로 표준화
  scaler = StandardScaler()
  df[['age','fare']] = scaler.fit_transform(df[['age','fare']]) #괄호가 하나면 Series 두개면 DataFrame
  # step4: 범주형 변수 인코딩
  df_encoded = pd.get_dummies(df, columns=['sex','embarked'],dtype = int)

  return df_encoded


print("자동화 전처리 완료")


In [ ]:
# 2단계 : 실제 데이터셋을 파이프라인에 통과시켜 완성하기

# 1. raw 데이터 불러오기
titanic_raw = sns.load_dataset('titanic')

# 2. 정답 데이터(y)와 입력데이터(x_raw) 분리
y = titanic_raw['survived']
X_raw = titanic_raw.drop(columns =['survived'])

# 3. 단 한줄로  모든 전처리 자동 실행
X_processed = pipeline_titanic_preprocessing(X_raw)

print("=========원본 데이터 상위 5행===============")
print(X_raw.head())
print(X_raw[['pclass','sex','age','fare','embarked']].head())

print("======= 전처리 파이프라인 완료 후 최종 머신러닝 입력 데이터 =======")
print(X_processed.head())
print("최종 데이터 구조:" , X_processed.shape)
print("결측치 총 개수: ", X_processed.isnull().sum().sum())